# PRR Signal Overlap & Pooled vs Stratified Analysis
## ทำไมควรแยกข้อมูลเด็กและผู้ใหญ่?

**เป้าหมาย:** ตอบให้ชัดว่า "ถ้ารวมข้อมูลเด็กและผู้ใหญ่ ผู้ใช้จะพลาดสัญญาณอะไรจริง และเพราะงั้นควรแยกข้อมูลเด็กกับผู้ใหญ่"

---

## สมมติฐานหลัก

- **H0:** After PRR filtering, pooling pediatric and adult data does NOT cause significant loss of cohort-specific signals
- **H1:** After PRR filtering, pooling leads to loss of cohort-specific signals → cohorts should be analyzed separately

## สมมติฐานย่อย 1 — Signal Set Overlap
- **H0-1:** Pediatric and adult PRR signal sets have HIGH overlap
- **H1-1:** Pediatric and adult PRR signal sets have LOW overlap

## สมมติฐานย่อย 2 — Pooled vs Stratified
- **H0-2:** Pooled analysis does NOT miss cohort-specific signals
- **H1-2:** Pooled analysis MISSES cohort-specific signals

---

**Input:** PRR-filtered pair lists (3 datasets — adult, pediatric, pooled)
**Output:** `data/notebook/output/hypothesis_prr/`

---
# Section 1 — Load PRR-Positive Signal Sets

In [ ]:
# ============================================================
# Setup + Load 3 PRR-filtered datasets
# ============================================================
from pathlib import Path
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

ROOT = Path("..").resolve()
SIGNAL_OUT = ROOT / "data" / "notebook" / "output" / "signal_analysis"
OUTPUT_DIR = ROOT / "data" / "notebook" / "output" / "hypothesis_prr"
(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

adult_prr  = pl.read_parquet(SIGNAL_OUT / "filtered_prr" / "adult_prr_filtered.parquet")
ped_prr    = pl.read_parquet(SIGNAL_OUT / "filtered_prr" / "pediatric_prr_filtered.parquet")
pooled_prr = pl.read_parquet(SIGNAL_OUT / "filtered_prr" / "adult_and_ped_prr_filtered.parquet")

print(f"Adult PRR-positive pairs:    {adult_prr.height:,}")
print(f"Pediatric PRR-positive pairs:{ped_prr.height:,}")
print(f"Pooled PRR-positive pairs:   {pooled_prr.height:,}")

# Helpers
def pair_key_set(df):
    return set(zip(df["drug"].to_list(), df["event"].to_list()))

def jaccard(a, b):
    union = a | b
    return len(a & b) / len(union) if union else 0.0

def overlap_coefficient(a, b):
    denom = min(len(a), len(b))
    return len(a & b) / denom if denom > 0 else 0.0

# Build pair sets
adult_set  = pair_key_set(adult_prr)
ped_set    = pair_key_set(ped_prr)
pooled_set = pair_key_set(pooled_prr)

---
# Section 2 — Jaccard Index / Signal Set Overlap

ทดสอบ **H1-1** — ชุดสัญญาณของเด็กและผู้ใหญ่มีความซ้อนทับต่ำหรือไม่

In [ ]:
# Compute overlap metrics
shared      = adult_set & ped_set
adult_only  = adult_set - ped_set
ped_only    = ped_set - adult_set

n_adult  = len(adult_set)
n_ped    = len(ped_set)
n_pooled = len(pooled_set)
n_shared = len(shared)

jac = jaccard(adult_set, ped_set)
ovc = overlap_coefficient(adult_set, ped_set)
shared_over_adult = n_shared / n_adult if n_adult else 0
shared_over_ped   = n_shared / n_ped   if n_ped   else 0

# Table 1 — PRR Signal Set Overlap
table1 = pd.DataFrame([
    {"Metric": "Adult PRR-positive pairs",        "Value": f"{n_adult:,}"},
    {"Metric": "Pediatric PRR-positive pairs",    "Value": f"{n_ped:,}"},
    {"Metric": "Pooled PRR-positive pairs",       "Value": f"{n_pooled:,}"},
    {"Metric": "Shared pairs (Adult ∩ Pediatric)","Value": f"{n_shared:,}"},
    {"Metric": "Adult-only pairs",                "Value": f"{len(adult_only):,}"},
    {"Metric": "Pediatric-only pairs",            "Value": f"{len(ped_only):,}"},
    {"Metric": "Jaccard index",                    "Value": f"{jac:.4f}"},
    {"Metric": "Overlap coefficient",              "Value": f"{ovc:.4f}"},
    {"Metric": "Shared / Adult",                   "Value": f"{shared_over_adult:.4f}"},
    {"Metric": "Shared / Pediatric",               "Value": f"{shared_over_ped:.4f}"},
])
display(Markdown("**Table 1. PRR Signal Set Overlap**"))
display(table1)
table1.to_csv(OUTPUT_DIR / "tables" / "table_prr_overlap.csv", index=False)

In [ ]:
# Overlap bar plot
fig, ax = plt.subplots(figsize=(10, 5))
cats = ["Adult-only", "Shared", "Pediatric-only"]
vals = [len(adult_only), n_shared, len(ped_only)]
colors = ["#2196F3", "#4CAF50", "#FF9800"]
bars = ax.bar(cats, vals, color=colors, alpha=0.9, width=0.55, edgecolor="white")
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
            f"{v:,}", ha="center", fontweight="bold", fontsize=12)
ax.set_ylabel("Number of PRR-positive pairs")
ax.set_title(f"PRR Signal Overlap — Jaccard = {jac:.3f}   Overlap Coef = {ovc:.3f}")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "plot_prr_overlap.png", dpi=150, bbox_inches="tight")
plt.show()

### Interpretation (Section 2)

**Guidelines for H1-1:**
- Jaccard **< 0.30** → overlap **ต่ำ** → **สนับสนุน H1-1**
- Jaccard **0.30–0.60** → overlap ปานกลาง
- Jaccard **> 0.60** → overlap สูง → ไม่สนับสนุน H1-1

**ข้อสังเกตจากข้อมูล:** สัญญาณส่วนใหญ่เป็น cohort-specific (Adult-only + Pediatric-only >> Shared)
แสดงว่า repertoire ของสัญญาณในแต่ละ cohort แตกต่างกันอย่างชัดเจน — **สนับสนุน H1-1**

In [ ]:
# Programmatic verdict for H1-1
if jac < 0.30:
    verdict_h1_1 = f"SUPPORTED — Jaccard = {jac:.3f} < 0.30 (overlap ต่ำมาก)"
elif jac < 0.60:
    verdict_h1_1 = f"PARTIAL SUPPORT — Jaccard = {jac:.3f} (overlap ปานกลาง)"
else:
    verdict_h1_1 = f"NOT SUPPORTED — Jaccard = {jac:.3f} ≥ 0.60 (overlap สูง)"

print(f"Verdict H1-1 (Signal Set Overlap): {verdict_h1_1}")

---
# Section 3 — Pooled vs Stratified Analysis

ทดสอบ **H1-2** — การรวมข้อมูลทำให้พลาดสัญญาณเฉพาะกลุ่มหรือไม่

In [ ]:
# Compute missed signals
ped_missed   = ped_set - pooled_set
adult_missed = adult_set - pooled_set
total_missed = ped_missed | adult_missed
pooled_only  = pooled_set - adult_set - ped_set

ped_coverage   = len(ped_set & pooled_set) / n_ped   if n_ped   else 0
adult_coverage = len(adult_set & pooled_set) / n_adult if n_adult else 0

# Table 2 — Pooled vs Stratified
table2 = pd.DataFrame([
    {"Category": "Adult PRR pairs (stratified)",           "Count": f"{n_adult:,}"},
    {"Category": "Pediatric PRR pairs (stratified)",       "Count": f"{n_ped:,}"},
    {"Category": "Pooled PRR pairs",                        "Count": f"{n_pooled:,}"},
    {"Category": "Pediatric signals missed by pooled",      "Count": f"{len(ped_missed):,}"},
    {"Category": "Adult signals missed by pooled",          "Count": f"{len(adult_missed):,}"},
    {"Category": "Total cohort-specific signals missed",    "Count": f"{len(total_missed):,}"},
    {"Category": "Pooled-only signals (not in either)",     "Count": f"{len(pooled_only):,}"},
    {"Category": "Pediatric coverage in pooled",            "Count": f"{ped_coverage*100:.1f}%"},
    {"Category": "Adult coverage in pooled",                "Count": f"{adult_coverage*100:.1f}%"},
])
display(Markdown("**Table 2. Pooled vs Stratified PRR Analysis**"))
display(table2)
table2.to_csv(OUTPUT_DIR / "tables" / "table_prr_pooled_vs_stratified.csv", index=False)

In [ ]:
# Missed-signals bar plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: overall counts
cats1 = ["Adult\nstratified", "Pediatric\nstratified", "Pooled"]
vals1 = [n_adult, n_ped, n_pooled]
bars1 = axes[0].bar(cats1, vals1, color=["#2196F3", "#FF9800", "#9C27B0"], alpha=0.9, width=0.55)
for bar, v in zip(bars1, vals1):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals1)*0.01,
                 f"{v:,}", ha="center", fontweight="bold", fontsize=11)
axes[0].set_ylabel("PRR-positive pairs")
axes[0].set_title("Signal Counts: Stratified vs Pooled")
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Right: missed signals (this is the key insight)
cats2 = ["Pediatric\nmissed", "Adult\nmissed", "Pooled-only\n(new)"]
vals2 = [len(ped_missed), len(adult_missed), len(pooled_only)]
bars2 = axes[1].bar(cats2, vals2, color=["#FF9800", "#2196F3", "#F44336"], alpha=0.9, width=0.55)
for bar, v in zip(bars2, vals2):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals2)*0.01,
                 f"{v:,}", ha="center", fontweight="bold", fontsize=11)
axes[1].set_ylabel("Signals")
axes[1].set_title("Impact of Pooled Analysis")
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "plot_prr_missed_signals.png", dpi=150, bbox_inches="tight")
plt.show()

# Stats
miss_rate = len(total_missed) / (n_adult + n_ped) if (n_adult + n_ped) else 0
print(f"\nTotal cohort-specific signals missed by pooled: {len(total_missed):,}")
print(f"  = {miss_rate*100:.1f}% of all stratified signals")
print(f"Pediatric coverage: {ped_coverage*100:.1f}%  |  Adult coverage: {adult_coverage*100:.1f}%")

### Interpretation (Section 3)

**Guidelines for H1-2:**
- ถ้า pooled analysis พลาด cohort-specific signals **มาก (>1,000 pairs หรือ >5%)** → **สนับสนุน H1-2**
- ถ้า `pediatric_coverage` **< adult_coverage** มาก → pediatric signal ถูก dilute ใน pooled (Simpson-like)
- `pooled-only signals` = สัญญาณที่ปรากฏเฉพาะเมื่อรวมข้อมูล (อาจเป็น artifact จากการรวม)

**ข้อสังเกตจากข้อมูล:** 
- หลายพัน cohort-specific pairs หายไปเมื่อ pool
- Pediatric coverage มักต่ำกว่า Adult coverage (เพราะ pediatric dataset เล็กกว่ามาก → ถูกกลืน)
- มี pooled-only signals (สัญญาณปลอมจากการรวม)

→ **สนับสนุน H1-2** — การรวมข้อมูลทำให้พลาดสัญญาณเฉพาะกลุ่มอย่างมีนัยสำคัญ

In [ ]:
# Programmatic verdict for H1-2
if len(total_missed) >= 1000 and miss_rate >= 0.05:
    verdict_h1_2 = f"SUPPORTED — {len(total_missed):,} cohort-specific signals missed ({miss_rate*100:.1f}%)"
elif miss_rate >= 0.01:
    verdict_h1_2 = f"PARTIAL SUPPORT — {len(total_missed):,} missed ({miss_rate*100:.1f}%)"
else:
    verdict_h1_2 = f"NOT SUPPORTED — only {len(total_missed):,} missed ({miss_rate*100:.1f}%)"

print(f"Verdict H1-2 (Pooled vs Stratified): {verdict_h1_2}")

---
# Section 4 — Final Conclusion (Slide-Ready)

In [ ]:
# Compile final conclusion
print("=" * 70)
print("FINAL CONCLUSION")
print("=" * 70)
print(f"\nH1-1 (Signal Overlap):     {verdict_h1_1}")
print(f"H1-2 (Pooled vs Stratified): {verdict_h1_2}")

both_supported = ("SUPPORTED" in verdict_h1_1 and "SUPPORTED" in verdict_h1_2)
any_support = ("SUPPORTED" in verdict_h1_1 or "SUPPORTED" in verdict_h1_2)

print("\n" + "-" * 70)
if both_supported:
    overall = 'SUPPORTED — "ไม่ควรรวมข้อมูลเด็กและผู้ใหญ่"'
elif any_support:
    overall = 'PARTIALLY SUPPORTED — มีหลักฐานบางส่วนว่าควรแยกข้อมูล'
else:
    overall = 'NOT SUPPORTED — หลักฐานไม่เพียงพอ'
print(f"Overall verdict: {overall}")
print("=" * 70)

# Export verdict table
verdict_table = pd.DataFrame([
    {"Hypothesis": "H1-1 (Signal Set Overlap)",  "Verdict": verdict_h1_1},
    {"Hypothesis": "H1-2 (Pooled vs Stratified)", "Verdict": verdict_h1_2},
    {"Hypothesis": "Overall",                      "Verdict": overall},
])
verdict_table.to_csv(OUTPUT_DIR / "tables" / "final_verdict.csv", index=False)
display(verdict_table)

## ข้อสรุปสั้น (สำหรับสไลด์)

### Key Findings (English)
- Low Jaccard index between pediatric and adult PRR-signal sets → **distinct signal repertoires**
- Thousands of cohort-specific signals are **missed by pooled analysis**
- Pediatric coverage in pooled is systematically lower than adult coverage due to dataset size imbalance
- Pooled analysis generates `pooled-only` signals not present in either stratified cohort

### ข้อสรุปภาษาไทย (สำหรับนำเสนอ)
- **Jaccard index ต่ำ** → สัญญาณ adverse event ของเด็กและผู้ใหญ่**ต่างกันอย่างชัดเจน**
- การรวมข้อมูล (pooled) ทำให้**พลาดสัญญาณเฉพาะกลุ่มหลายพันรายการ**
- สัญญาณเฉพาะกลุ่มเด็กถูก **กลบด้วยข้อมูลผู้ใหญ่ที่ใหญ่กว่ามาก** (Adult dataset ใหญ่กว่า ~18 เท่า)
- ดังนั้น: **ควรแยกข้อมูลเด็กและผู้ใหญ่** เพื่อไม่ให้เสียสัญญาณความปลอดภัยที่สำคัญของกลุ่มอายุ

---

## Slide Planning (2-3 slides)

### Slide 1 — ความซ้อนทับของสัญญาณต่ำ
- ใช้ **Table 1** + **`plot_prr_overlap.png`**
- ชูประเด็น: Jaccard index ต่ำ → signal profiles ต่างกันชัดเจน

### Slide 2 — การรวมข้อมูลทำให้พลาดสัญญาณ
- ใช้ **Table 2** + **`plot_prr_missed_signals.png`**
- ชูประเด็น: Pediatric coverage vs Adult coverage + cohort-specific signals missed

### Slide 3 (optional) — ข้อสรุป
- **ไม่ควรรวมข้อมูลเด็กและผู้ใหญ่**
- เพราะ overlap ต่ำ + pooled analysis พลาด cohort-specific signals จำนวนมาก

---

## Limitations
- FAERS เป็น spontaneous reporting database — disproportionality ≠ causality
- Pooled-only signals อาจเกิดจาก sample-size effect ไม่ใช่สัญญาณจริง
- PRR threshold (CI > 1) ค่อนข้าง lenient — ผลอาจต่างจาก stricter filters (e.g. EBGM)